In [ ]:
from huggingface_hub import login
from google.colab import drive
import pandas as pd
from PIL import Image
import torch
from diffusers import FluxPipeline

In [ ]:
# login huggingface and mount drive
login()
drive.mount('/content/drive')

In [ ]:
images_description_file_path = "/content/drive/MyDrive/Thesis/tobegenerated.xlsx"
df_images_descriptions = pd.read_excel(images_description_file_path)

In [ ]:
# create flux pipeline from huggingface
pipe = FluxPipeline.from_pretrained("black-forest-labs/FLUX.1-dev", torch_dtype=torch.bfloat16)
pipe.enable_model_cpu_offload()

In [ ]:
# Store generated images here
output_base_path = "/content/drive/My Drive/Thesis/images_generated/"
# Number of images to generate per original path
num_images = 5

for index, image_path_str in df_images_descriptions['image_path'].items():
    # get prompt from caption
    prompt = df_images_descriptions['caption'][index]
    # create image_name based on original image path
    parts = image_path_str.strip().lstrip('./').split('/')
    image_name_path_based = f"{parts[0]}_{parts[1]}_{parts[2]}_{parts[3].split('.')[0]}"
    for i in range(num_images):
        # Create a new generator with a different seed
        generator = torch.Generator("cpu").manual_seed(i)

        # Generate the image
        image = pipe(
            prompt,
            height=1024,
            width=1024,
            guidance_scale=3.5,
            num_inference_steps=50,
            max_sequence_length=512,
            generator=generator
        ).images[0]

        # Save the image
        output_path = f"{output_base_path}{image_name_path_based}_{i + 1}.jpg"
        image.save(output_path)
        print(f"Image saved at: {output_path}")